### Step 1: Importing Necessary Libraries

In [2]:
import pandas as pd
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import Counter

In [3]:
nltk.download('vader_lexicon')
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\cfl602\AppData\Roaming\nltk_data...
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\cfl602\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\cfl602\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [5]:
df = pd.read_csv("../DataSets/amazon.csv")
print("Dataset shape:", df.shape)
df.head()

Dataset shape: (20000, 2)


,reviewText,Positive
0,This is a one of the best apps acording to a b...,1
1,This is a pretty good version of the game for ...,1
2,this is a really cool game. there are a bunch ...,1
3,"This is a silly game and can be frustrating, b...",1
4,This is a terrific game on any pad. Hrs of fun...,1


### Step 2: Feature Engineering (BoW)

In [6]:
def preprocess_text(text):
    """Tokenize, lower, remove stopwords/punctuation, return a clean list of words."""
    tokens = word_tokenize(str(text).lower())
    # Remove non‑alphabetic tokens and stopwords
    clean_tokens = [word for word in tokens if word.isalpha() and word not in stopwords.words('english')]
    return clean_tokens

In [7]:
all_words = []
for review in df['reviewText']:
    all_words.extend(preprocess_text(review))

In [8]:
vocabulary = set(all_words)
print(f"Total vocabulary size: {len(vocabulary)}")
print(f"Sample vocabulary: {list(vocabulary)[:20]}")

Total vocabulary size: 17770
Sample vocabulary: ['delivery', 'certianly', 'monroe', 'teens', 'chowder', 'logo', 'apps', 'spirit', 'merely', 'bracket', 'ready', 'doll', 'front', 'dug', 'celtonasreally', 'heights', 'cou', 'elementry', 'togeather', 'captures']


### Step 3: Categorisation logic

In [9]:
positive_terms = {'good', 'great', 'excellent', 'amazing', 'love', 'awesome', 'perfect', 'wonderful'}
negative_terms = {'bad', 'poor', 'slow', 'terrible', 'awful', 'waste', 'horrible', 'worst'}

### Step 4: Feature extraction – scan BoW vectors

In [10]:
analyzer = SentimentIntensityAnalyzer()

In [11]:
def get_vader_sentiment(text):
    """Return 1 if compound score >= 0.05, else 0."""
    scores = analyzer.polarity_scores(str(text))
    return 1 if scores['compound'] >= 0.05 else 0


In [12]:
df['vader_sentiment'] = df['reviewText'].apply(get_vader_sentiment)

In [ ]:
def count_terms(review):
    tokens = preprocess_text(review)
    pos_count = sum(1 for w in tokens if w in positive_terms)
    neg_count = sum(1 for w in tokens if w in negative_terms)
    return pos_count, neg_count

df[['pos_term_count', 'neg_term_count']] = df['reviewText'].apply(
    lambda x: pd.Series(count_terms(x))
)

print("\nFirst 5 reviews with BoW term counts & VADER prediction:")
print(df[['reviewText', 'pos_term_count', 'neg_term_count', 'vader_sentiment']].head())

### Step 5: Performance report – % positives & negatives

In [ ]:
total_reviews = len(df)
pos_reviews = df['vader_sentiment'].sum()
neg_reviews = total_reviews - pos_reviews

print("\n" + "="*50)
print("FINAL PERFORMANCE REPORT")
print("="*50)
print(f"Total reviews analysed: {total_reviews}")
print(f"Positive reviews (VADER): {pos_reviews} ({pos_reviews/total_reviews*100:.2f}%)")
print(f"Negative reviews (VADER): {neg_reviews} ({neg_reviews/total_reviews*100:.2f}%)")
print("\nTerm‑based BoW indicators:")
print(f"Reviews containing positive terms: {(df['pos_term_count'] > 0).sum()} ({(df['pos_term_count'] > 0).mean()*100:.2f}%)")
print(f"Reviews containing negative terms: {(df['neg_term_count'] > 0).sum()} ({(df['neg_term_count'] > 0).mean()*100:.2f}%)")